# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
%pip install -Uqqq langchain langchain-openai langchain-community

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

Simple chain

In [3]:
from langchain_core.prompts import PromptTemplate  # 프롬프트 체인 구성 시 사용
from langchain.chat_models import init_chat_model  # 모델 체인 구성 래퍼 클래스
from langchain_core.output_parsers import StrOutputParser  # 답변 문자열 반환

prompt = PromptTemplate.from_template("{city}의 특산물은 무엇입니까?")
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('홍콩'))  # 프롬프트 템플릿 값이 1개일때만 사용
# print(chain.invoke(input={'city': '홍콩', ...}))  # 프롬프트 템플릿 변수가 2개 이상일 경우 dict형으로 전달

홍콩의 대표적인 특산물과 먹거리는 다음과 같습니다.

- **딤섬(點心)**: 하가우, 샤오마이, 차슈바오 등 다양한 광둥식 만두와 간식  
- **에그타르트**: 바삭한 페이스트리와 달콤한 커스터드가 어우러진 디저트  
- **파인애플 번**: 파인애플이 들어간 것이 아니라, 겉면이 바삭하고 무늬가 파인애플처럼 생긴 빵  
- **홍콩식 밀크티**: 진하고 부드러운 맛의 홍콩식 차  
- **로스트 구스**: 겉은 바삭하고 속은 촉촉한 광둥식 거위구이  
- **차슈**: 달콤한 양념으로 구운 돼지고기  
- **완탕면·소고기 국수**: 홍콩식 면 요리  
- **해산물과 건어물**: 말린 전복, 해삼, 새우, 관자 등  
- **중국차**: 보이차, 철관음, 우롱차 등  
- **홍콩식 과자와 소스**: 아몬드 쿠키, 참깨 쿠키, 굴소스, XO소스 등  

기념품으로는 **중국차, 홍콩식 과자, 굴소스·XO소스, 전통 약재와 건어물**이 많이 판매됩니다.


In [4]:
print(chain.invoke(input={'city': '마카오'}))

마카오의 대표적인 특산물과 먹거리는 다음과 같습니다.

- **에그타르트(포르투갈식 타르트)**: 바삭한 페이스트리와 부드러운 커스터드가 특징입니다.
- **아몬드 쿠키**: 고소하고 담백한 전통 과자로, 선물용으로 인기가 많습니다.
- **육포·돼지고기 육포**: 달콤하고 짭짤하게 양념한 마카오식 육포입니다.
- **돼지갈비빵(주파빠우)**: 바삭한 빵에 양념한 돼지갈비를 넣은 간식입니다.
- **세라두라**: 포르투갈계 디저트로, 크림과 잘게 부순 비스킷을 층층이 쌓아 만듭니다.
- **포르투갈식 요리**: 아프리칸 치킨, 바칼라우(염장 대구), 해산물 요리 등이 유명합니다.
- **생강 사탕과 전통 과자**: 기념품으로 많이 구매하는 마카오식 간식입니다.

특히 여행 기념품으로는 **에그타르트, 아몬드 쿠키, 육포**가 가장 대표적입니다.


Sequential Chain

In [5]:
prompt1 = PromptTemplate.from_template("다음 내용을 한글로 번역하세요. {eng_text}")
prompt2 = PromptTemplate.from_template("다음 내용을 요약하세요. {kor_text}")

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm

eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""
print(chain1.invoke(eng_text))

# 요약 체인
chain2 = prompt2 | llm | output_parser

kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""
print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계를 극복하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.\n\n이를 위해 먼저 문서 로더(document loader)를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 97, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHHwDjYa3Qr1aFtvaOSZkivWt1SLb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0408a-e02f-78e0-abac-9459f0a3c531-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={

In [6]:
# Sequential Chain
chain = chain1 | chain2
print(chain.invoke({'eng_text': eng_text}))

LLM은 특정 문서나 이메일 등 외부 맥락에 직접 접근하지 못하는 한계가 있습니다. 이를 해결하려면 외부 데이터를 LLM에 연결해야 하며, LangChain에서는 문서 로더를 통해 PDF, 이메일, 웹사이트, YouTube 동영상 등 다양한 자료를 불러올 수 있습니다.


Conditional Chain

In [7]:
from langchain_core.runnables import RunnableBranch  # 조건에 따라 체인을 분기 실행해주는 Runnable

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 중간 풀이과정과 함께 작성해주세요. {question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

# math_chain을 선택할 수 있는 함수 (질문에 계산 또는 calc가 포함되면 수학 체인 선택)
def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')  # 입력받은 딕셔너리 dict에서 question 키의 값을 추출 (없으면 빈 문자열)
    return '계산' in question or 'calc' in question

# 분기 체인
branch_chain = RunnableBranch(
    (is_math_question, math_chain),  # True/False 결과 조건이 True면 math_chain
    default_chain  # False면 default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산해줘.'}))

단계별로 계산하면:

1. \(125 \times 3 = 375\)
2. \(375 + 50 = 425\)

따라서 정답은 **425**입니다.


In [8]:
print(branch_chain.invoke({'question': '나 오늘 우울해. 빵? 밥? 뭘 먹을지 계산이 안 서'}))

오늘은 **밥**으로 가자. 우울한 날엔 따뜻하고 든든한 음식이 결정 피로를 줄여줘.

### 간단한 계산
1. **기분이 가라앉음** → 따뜻한 음식에 +2점  
2. **든든함이 필요함** → 밥에 +2점  
3. **준비하기 귀찮음** → 빵에 +2점  
4. 오늘은 우울하다고 했으니, 든든함과 따뜻함을 우선

**결론: 밥 + 계란이나 김 + 국/찌개**  
거창하게 차릴 필요 없이 편의점 삼각김밥이나 즉석밥도 충분해. 오늘의 목표는 “완벽한 식사”가 아니라 **일단 조금 먹고 몸을 챙기기**야.

그리고 우울한 마음이 단순히 기분이 안 좋은 정도를 넘어서 너무 버겁거나, 스스로를 해치고 싶은 생각까지 든다면 혼자 버티지 말고 가까운 사람이나 전문 상담기관에 바로 연락해줘.


In [16]:
###########이 이후로 ~깃허브~

Memory Chain

`RunnableWithMessageHistory`를 사용하여 대화내역을 기억하는 chain을 생성한다

In [12]:
from langchain_core.chat_history import BaseChatMessageHistory  # LANGCHAIN 대화기록 메모리 저장용 클래스
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage  # 메시지 타입들
from pydantic import BaseModel, Field  # Pydantic 모델 (검증/기본값 생성) 도구
from typing import List  # 타입 힌트 (List)

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    # Field(default_factory=list) : 인스턴스마다 독립적인 messages list를 구성
    messages: List[BaseMessage] = Field(default_factory=list)

    def add_message(self, message: List[BaseMessage]) -> None:
        self.messages.extend(message)  # 전달받은 메시지들을 기존 리스트 뒤에 추가

    def clear(self) -> None:
        self.messages = []  # 지정된 메시지들을 초기화

store = {}  # {session_id: 히스토리 객체(InMemoryHistory)} 저장소

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()  # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]  # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1')  # 세션 ID '1'의 히스토리 가져오기 (없으면 메모리 공간 생성)
history1.add_message([AIMessage(content='반갑습니다, Capybara님.')])  # AI 메시지 추가
history1.add_message([HumanMessage(content='그래, 나 cap이야. 만나서 반갑다.')])  # 유저메시지 추가
print(f'{history1 = }')  # f'history1 = {history1}'

history2 = get_by_session_id('2')  # 세션 ID '2'의 히스토리 가져오기
print(f'{history2 = }')

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다, Capybara님.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래, 나 cap이야. 만나서 반갑다.', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # 채팅 프롬프트 템플릿 / 히스토리 자리표시자
from langchain_core.runnables import RunnableWithMessageHistory  # 실행시 히스토리를 보여주는 Runnable

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),  # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,  # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question',  # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key='history'  # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({  
    'domain' : 'math', 
    'question' : '민수는 강아지를 3마리 키우고 있습니다.'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable':{
        'session_id' : '100'  # 어떤 세션 히스토리 사용할지 
    }
})

c:\Users\Playdata\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='그렇군요. 민수는 강아지 세 마리를 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 38, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 23, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI94x9I9jpTqKRNw8GtO6YFxekcU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04097-09cb-7c91-a33b-7baba87f8811-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 52, 'total_tokens': 90, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 23}})

In [15]:
chain_with_history.invoke({  
    'domain' : 'math', 
    'question' : '소라는 고양이를 4마리 키우고 있습니다.'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable':{
        'session_id' : '100'  # 어떤 세션 히스토리 사용할지 
    }
})

ValueError: Unexpected message type: 'content'. Use one of 'human', 'user', 'ai', 'assistant', 'function', 'tool', 'system', or 'developer'.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_COERCION_FAILURE 

In [ ]:
store  # 현재 메모리에 저장된 세션별 대화 히스토리

ChatMessageHistory

In [17]:
store = {}  # {session_id: 히스토리 객체(InMemoryHistory)} 저장소

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = RunnableWithMessageHistory()  # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]  # 해당 세션의 히스토리 객체 반환

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),  # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm | output_parser

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,  # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question',  # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key='history'  # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({  
    'domain' : '심리상담', 
    'question' : '요즘 너무 더워서 갑자기 너무 짜증나는데? 왜이러지?'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable':{
        'session_id' : '100'  # 어떤 세션 히스토리 사용할지 
    }
})

c:\Users\Playdata\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


TypeError: RunnableWithMessageHistory.__init__() missing 2 required positional arguments: 'runnable' and 'get_session_history'

In [18]:
store

{}

##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리